[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-08b-cross-attention.ipynb)

# Lecture 8b: where does a Transformer look while it translates?

Lecture 8b computed one attention step by hand. Here you write that step as a function, check it
against a trained Transformer's own, and then use it to see where the model looks in an English
sentence while it writes each piece of the Spanish one.

By the end you will be able to:

1. compute one attention step (query, key, value, scaled dot product, softmax) and show it is the
   step a trained Transformer takes, head by head;
2. name a Transformer's three attentions, say what each is for, and give the shape of each one's
   weights;
3. read cross-attention as an alignment map, on a sentence the model never saw in training;
4. tell finding the right source words from translating them well, and say which one failed.

These serve Lecture 8b's outcome *Describe self-attention (query, key, value, scaled dot
product, softmax) and compute one step by hand; say why there are three attentions and what each
is for; read a tensor shape*. The step you write is the same in all three attentions; this
notebook runs it as cross-attention.

You write one function, marked TODO, and a check tells you when it is right; then a prediction,
which stops the notebook until you write it, before the map that tests it. Everything runs on a
CPU in under a minute. Lecture 8c's notebook continues with the same model, one head at a time.

**Running in Google Colab?** Run the cells in order. The second downloads the trained model,
about 11 MB. There is nothing to uncomment, and no GPU is needed.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — the in-class activity for Lecture 8b (*Neural MT: The Transformer*).

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo

In [ ]:
from torchlingo.colab import setup

# The trained model, its subword vocabulary and its held-out test set. A pip install does
# not ship them, so setup downloads any that are missing.
env = setup(data=["data/pretrained/model.pt", "data/pretrained/spm.model", "data/pretrained/test.tsv"])

import math

import torch

from torchlingo.config import get_default_config
from torchlingo.visualization import format_attention

CFG = get_default_config()

## Step 1: one attention step

Attention lets each position gather information from other positions. Each position that is
looking asks a **query**; each position it may look at offers a **key**, which the query is
compared with, and a **value**, which is what gets passed on. One step:

1. score each query against every key with a dot product, divided by √d_k, where d_k is the
   length of one key;
2. turn each query's row of scores into weights with softmax, so the row sums to 1;
3. return, for each query, the weighted sum of the values.

In symbols, softmax(QKᵀ / √d_k) V. Here `q` is (queries, d_k), `k` is (keys, d_k) and `v` is
(keys, d_v). Return both the weights, (queries, keys), and the output, (queries, d_v): the
weights are the map you read later.

**Yours to write: `attention`.** `@` multiplies matrices, `.T` transposes one, and
`torch.softmax(x, dim=-1)` applies softmax along each row. The cell checks it on a case small
enough to work on paper, which is worth doing first: then a failed check tells you something.

In [ ]:
def attention(q, k, v):
    """Return (weights, output) for one attention step: softmax(q kᵀ / √d_k) v.

    TODO: write it. q is (queries, d_k), k is (keys, d_k), v is (keys, d_v). weights is
    (queries, keys), each row summing to 1; output is (queries, d_v).
    """
    raise NotImplementedError("write attention, then run this cell again")


# A case to work on paper: one query, two keys of length 2, and values easy to tell apart.
q = torch.tensor([[1.0, 0.0]])
k = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
v = torch.tensor([[10.0, 0.0], [0.0, 20.0]])
weights, output = attention(q, k, v)

assert tuple(weights.shape) == (1, 2) and tuple(output.shape) == (1, 2), (
    f"weights should be (queries, keys) = (1, 2) and output (queries, d_v) = (1, 2); "
    f"got {tuple(weights.shape)} and {tuple(output.shape)}"
)
assert torch.allclose(weights.sum(dim=-1), torch.ones(1)), (
    "each row of weights must sum to 1: apply softmax across the keys, dim=-1"
)
assert not torch.allclose(weights, torch.tensor([[0.7311, 0.2689]]), atol=1e-3), (
    "these are the weights before scaling: divide the scores by √d_k before the softmax"
)
assert torch.allclose(weights, torch.tensor([[0.6698, 0.3302]]), atol=1e-3), (
    f"expected weights [0.670, 0.330]; got {weights.tolist()}"
)
assert torch.allclose(output, torch.tensor([[6.698, 6.604]]), atol=1e-2), (
    f"expected output [6.698, 6.604], the weighted sum of the values; got {output.tolist()}"
)

# Three queries, five keys, values of length 6: one row of weights per query.
weights, output = attention(torch.randn(3, 4), torch.randn(5, 4), torch.randn(5, 6))
assert tuple(weights.shape) == (3, 5) and tuple(output.shape) == (3, 6), (
    f"expected weights (3, 5) and output (3, 6); got {tuple(weights.shape)} and {tuple(output.shape)}"
)
print("attention passes its checks.")

## Step 2: the model, and a sentence it has never seen

The model is a small Transformer of the kind you train for A8: three encoder layers and three
decoder layers, trained on English–Spanish text. It reads **subword pieces** rather than words:
its vocabulary splits a less common word into more common parts, and `▁` marks a piece that
starts a word, so "report" becomes `▁re` + `port`.

The sentence comes from the model's held-out test set, so it never saw it in training. It
translates **greedily**: at each step it writes the single most likely next piece. The cell
prints the English, the model's Spanish, a human reference, and the pieces of each.

In [ ]:
from pathlib import Path

import pandas as pd

from torchlingo.checkpoint import load_model
from torchlingo.data_processing.vocab import SentencePieceVocab
from torchlingo.inference import greedy_decode

PRETRAINED = Path("data/pretrained")
SENTENCE = "This report is our main public report."

# The model file holds the weights and the settings the model was built with; its vocabulary is
# the separate spm.model, one for both languages. load_model returns the model in inference
# mode: no dropout, so attention rows sum to 1.
sp_vocab = SentencePieceVocab(str(PRETRAINED / "spm.model"))
transformer, _, _ = load_model(PRETRAINED / "model.pt", "cpu", src_vocab=sp_vocab)

held_out = pd.read_csv(PRETRAINED / "test.tsv", sep="\t", dtype=str, keep_default_na=False)
match = held_out[held_out.src == SENTENCE]
assert len(match) == 1, f"expected one row for {SENTENCE!r}, found {len(match)}"

source = torch.tensor([sp_vocab.encode(SENTENCE, add_special_tokens=True)])
produced = greedy_decode(transformer, source, max_len=30, config=CFG)[0]

source_pieces = sp_vocab.indices_to_tokens(source[0].tolist())
# The pieces the model wrote, without the <sos> it starts from.
output_pieces = sp_vocab.indices_to_tokens(list(produced))[1:]

print("EN   ", SENTENCE)
print("MODEL", sp_vocab.decode(produced, skip_special_tokens=True))
print("REF  ", match.tgt.iloc[0])
print()
print("Source pieces:", source_pieces)
print("Output pieces:", output_pieces)

## Step 3: your step against the model's own

A Transformer runs the same step in three places, and they differ only in where the queries,
keys and values come from, so in the shape of the weights:

| attention | queries from | keys and values from | weights |
|---|---|---|---|
| encoder self-attention | the source | the source | (source pieces, source pieces) |
| decoder masked self-attention | the output so far | the output so far, never a later piece | (output pieces, output pieces) |
| **cross-attention** | the output so far | the encoder's reading of the source | (output pieces, source pieces) |

The first lets each source piece read its sentence; the second lets each output piece read what
has been written before it; the third is where the translation looks at the source, and the
only one that relates the two languages. This notebook reads the third.

A layer makes queries, keys and values by multiplying its inputs by three learned matrices and
adding a learned bias: Q = X W_Q + b_Q from the decoder's state X, and K = M W_K + b_K and
V = M W_V + b_V from the encoder's output M.

This model has four **heads** in each layer, and each head uses its own slice of those
matrices: of the 128 numbers in each query, head 1 uses the first 32, head 2 the next 32, and so
on. Lecture 8c's notebook is about why. Here, each head is one call to your `attention`.

The first cell records X and M in the last decoder layer while the model re-reads the
translation it produced, along with the weights the model computed. The second makes Q, K and V,
runs your function once per head, and compares.

In [ ]:
# Setup: run this cell, no need to read it.
# It records what the last decoder layer's cross-attention received (the decoder state X and the
# encoder output M) and the weights it computed for each head, while the model re-reads its own
# translation. PyTorch computes the weights only when asked, hence the wrapper.
cross = transformer.transformer.decoder.layers[-1].multihead_attn
seen = {}


def recording(query, key, value, *args, _forward=cross.forward, **kwargs):
    kwargs.update(need_weights=True, average_attn_weights=False)
    out, weights = _forward(query, key, value, *args, **kwargs)
    seen.update(X=query[0], M=key[0], weights=weights[0])
    return out, weights


cross.forward = recording
try:
    with torch.no_grad():
        transformer(source, torch.tensor([produced]))
finally:
    del cross.forward

In [ ]:
d_model, n_heads = cross.embed_dim, cross.num_heads
d_k = d_model // n_heads
W_Q, W_K, W_V = cross.in_proj_weight.split(d_model)   # each (d_model, d_model)
b_Q, b_K, b_V = cross.in_proj_bias.split(d_model)

with torch.no_grad():
    Q = seen["X"] @ W_Q.T + b_Q   # (output pieces, d_model)
    K = seen["M"] @ W_K.T + b_K   # (source pieces, d_model)
    V = seen["M"] @ W_V.T + b_V

    yours = []
    for h in range(n_heads):
        part = slice(h * d_k, (h + 1) * d_k)   # this head's 32 columns
        weights, _ = attention(Q[:, part], K[:, part], V[:, part])
        gap = (weights - seen["weights"][h]).abs().max().item()
        print(f"head {h + 1}: largest difference from the model's own weights {gap:.1e}")
        assert gap < 1e-4, f"head {h + 1} differs from the model: recheck attention against Step 1"
        yours.append(weights)

print("Your attention step is the model's, in all four heads.")

## Step 4: predict, then look

The map two cells down averages your four heads. Each row is an output piece, each column a
source piece, and the darker the cell (`·` `░` `▒` `▓` `█`), the more weight that output piece
put on that source piece while the model wrote it. It is the alignment matrix of Lecture 8a,
attention as a learned soft alignment, read the same way, now made by a Transformer.

**Write a prediction first**, in `PREDICTION` below: choose three output pieces from Step 2's
list, and for each write the source piece you expect it to look at most, copied exactly as
printed (with its `▁`). The notebook stops there until you write it; the map's cell then marks
each one.

In [ ]:
PREDICTION = {}   # <- three output pieces, each with the source piece you expect, e.g. {"▁piece": "▁piece"}

assert PREDICTION, "Write your prediction in PREDICTION, then run this cell again."
assert len(PREDICTION) == 3, f"Choose three output pieces; you have {len(PREDICTION)}."
for piece, guess in PREDICTION.items():
    assert piece in output_pieces, f"{piece!r} is not an output piece: copy it from Step 2's list"
    assert guess in source_pieces, f"{guess!r} is not a source piece: copy it from Step 2's list"
print("Prediction recorded.")

In [ ]:
# Row n is the state that predicts output piece n; the last row, after <eos>, predicts nothing.
averaged = torch.stack(yours).mean(dim=0)[: len(output_pieces)]
print(format_attention(averaged, source_pieces, output_pieces))
print()

for piece, guess in PREDICTION.items():
    looked = source_pieces[int(averaged[output_pieces.index(piece)].argmax())]
    verdict = "right" if guess == looked else "not this time"
    print(f"{piece!r}: you predicted {guess!r}; it looked most at {looked!r} ({verdict})")

**Before you read on**, write one sentence in `MISSED`: which of your predictions missed, and
what that output piece looked at instead. If none missed, say which came closest to missing.

In [ ]:
MISSED = ""   # <- which prediction missed, and what that output piece looked at instead

print("What missed:", MISSED.strip() or "(none written)")

## Read it against the translation

The output is wrong: `Esta es nuestra pública de reporte.` against a reference of `Este reporte
es nuestro principal reporte público.` But look at where the attention went, because the obvious
conclusion is not the right one.

**Attention found the right words.** `▁Esta` peaks on `▁This`, `▁nuestra` on `▁our`, `▁públ` on
`▁public`, and `port` on the `port` piece of "report". Four content words, four correct
alignments, learned from nothing but examples, on a sentence the model has never seen, with a
vocabulary of pieces rather than words.

**So the failure is not in attention.** It is in what the decoder does with what attention
handed it. The model retrieves `public`, `our` and `report` and then assembles them into
`nuestra pública de reporte`: Spanish words in an order Spanish does not use, with `de` from
nowhere. Knowing *where to look* and knowing *what to write* are separate abilities, and this
model has more of the first than the second.

**Look at the word that disappeared.** "main" is split into `▁ma` and `in`, and those two columns
are the faintest in the map: no row peaks on either. "main" has no counterpart in the output. A
dropped word often shows up here first, as a column nothing looks at.

**The texture is diffuse**: a few `▓▓▓` and `▒▒▒` peaks in a wash of `░░░`, not one dark cell per
row. Part of that is real data, where a word rarely has exactly one counterpart. Part of it is
the averaging: you just saw that the map is the mean of four heads. Lecture 8c's notebook takes
them apart.